# 30-Day Readmission Risk Prediction

End-to-end ML use case: binary classification to predict 30-day readmission from encounter and patient features.
Uses the healthcare mock dataset (patients, encounters, diagnoses, labs, readmissions).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "python" else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "raw"

## Load data

In [ ]:
patients = pd.read_csv(DATA_DIR / "patients.csv")
encounters = pd.read_csv(DATA_DIR / "encounters.csv", parse_dates=["admit_date", "discharge_date"])
diagnoses = pd.read_csv(DATA_DIR / "diagnoses.csv")
labs = pd.read_csv(DATA_DIR / "labs.csv", parse_dates=["result_date"])
readmissions = pd.read_csv(DATA_DIR / "readmissions.csv", parse_dates=["index_discharge", "readmit_date"])
payers = pd.read_csv(DATA_DIR / "payers.csv")

## Build target and feature set (encounter-level)

Target: whether the encounter led to a 30-day readmission. Features: demographics, LOS, encounter type, diagnosis count, lab summary, payer.

In [ ]:
# Target: encounter has 30-day readmission
enc_with_readmit = set(readmissions[readmissions["is_30_day"] == 1]["encounter_id"])
encounters["readmit_30"] = encounters["encounter_id"].isin(enc_with_readmit).astype(int)

# LOS in days
encounters["los_days"] = (
    (encounters["discharge_date"] - encounters["admit_date"]).dt.total_seconds() / 86400
).clip(lower=0)

# Demographics: age at encounter
encounters["admit_date_d"] = pd.to_datetime(encounters["admit_date"])
patients["date_of_birth"] = pd.to_datetime(patients["date_of_birth"])
encounters = encounters.merge(patients[["patient_id", "date_of_birth", "gender", "payer_id"]], on="patient_id", how="left")
encounters["age"] = (
    (encounters["admit_date_d"] - encounters["date_of_birth"]).dt.days / 365.25
).clip(0, 120)

# Diagnosis count per encounter
diag_count = diagnoses.groupby("encounter_id").size().reset_index(name="diag_count")
encounters = encounters.merge(diag_count, on="encounter_id", how="left")
encounters["diag_count"] = encounters["diag_count"].fillna(0).astype(int)

# Lab: mean result_num per encounter (e.g. CMP)
lab_agg = labs.groupby("encounter_id")["result_num"].agg(["mean", "count"]).reset_index()
lab_agg.columns = ["encounter_id", "lab_mean", "lab_count"]
encounters = encounters.merge(lab_agg, on="encounter_id", how="left")
encounters["lab_mean"] = encounters["lab_mean"].fillna(encounters["lab_mean"].median())
encounters["lab_count"] = encounters["lab_count"].fillna(0).astype(int)

# Encounter type dummies
encounters = pd.get_dummies(encounters, columns=["encounter_type"], prefix="enc")

feature_cols = ["los_days", "age", "diag_count", "lab_mean", "lab_count"] + [
    c for c in encounters.columns if c.startswith("enc_")
]
X = encounters[feature_cols].fillna(0)
y = encounters["readmit_30"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

## Evaluation

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No readmit", "Readmit 30d"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print(confusion_matrix(y_test, y_pred))

In [ ]:
imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp.plot(kind="barh", title="Feature importance (readmission model)");

## Summary

Simple interpretable model for 30-day readmission using encounter and patient features. Can be extended with more diagnoses (comorbidity indices), prior utilization, and facility/payer.